In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import shapiro, spearmanr, kruskal
import warnings
warnings.filterwarnings('ignore')
import os

# Configuración de rutas
data_path = r"C:\Users\PC\Desktop\ProjecteData\Equip_15\Data\RRHH_220925_clean.parquet"
output_path = r"C:\Users\PC\Desktop\ProjecteData\Equip_15\Data\Resultados_Analisis_220925"

# Crear directorio de salida si no existe
os.makedirs(output_path, exist_ok=True)

# Configuración de estilo para gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("="*80)
print("ANÁLISIS ESTADÍSTICO DE AUSENTISMO LABORAL")
print("="*80)

# 1. CARGA Y VERIFICACIÓN DE DATOS
print("\n1. CARGANDO Y VERIFICANDO DATOS...")
print("-" * 50)

try:
    df = pd.read_parquet(data_path)
    print(f"✓ Dataset cargado correctamente: {df.shape[0]} filas, {df.shape[1]} columnas")
except Exception as e:
    print(f"✗ Error cargando el archivo: {e}")
    exit()

# 2. VERIFICACIÓN DE TIPOS DE DATOS
print("\n2. VERIFICACIÓN DE TIPOS DE DATOS...")
print("-" * 50)

# Mostrar tipos actuales
print("Tipos de datos actuales:")
for col in df.columns:
    print(f"  {col}: {df[col].dtype}")

# Definir tipos esperados para análisis
expected_types = {
    'Transportation_expense': 'numeric',
    'Distance_Residence_Work': 'numeric', 
    'Service_time': 'numeric',
    'Age': 'numeric',
    'Work_load_Average_day': 'numeric',
    'Hit_target': 'numeric',
    'Disciplinary_failure': 'numeric',
    'Son': 'numeric',
    'Social_drinker': 'numeric',
    'Social_smoker': 'numeric',
    'Pet': 'numeric',
    'Weight': 'numeric',
    'Height': 'numeric',
    'Body_mass_index': 'numeric',
    'Education_numeric': 'numeric',
    'Month_absence_order': 'numeric',
    'Day_week_order': 'numeric',
    'Seasons_order': 'numeric',
    'Absenteeism_hours': 'numeric',
    'Month_absence': 'categorical',
    'Day_week': 'categorical',
    'Seasons': 'categorical',
    'Education': 'categorical'
}

# Convertir tipos si es necesario
conversion_issues = []
for col, expected_type in expected_types.items():
    if col in df.columns:
        if expected_type == 'numeric':
            if not pd.api.types.is_numeric_dtype(df[col]):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    conversion_issues.append(f"{col} → numérico")
                except:
                    print(f"✗ No se pudo convertir {col} a numérico")
        elif expected_type == 'categorical':
            if not pd.api.types.is_string_dtype(df[col]):
                df[col] = df[col].astype(str)

if conversion_issues:
    print("✓ Conversiones realizadas:", conversion_issues)
else:
    print("✓ Todos los tipos de datos son correctos")

# 3. ELIMINAR VARIABLES NO RELEVANTES
print("\n3. PREPARACIÓN DE VARIABLES...")
print("-" * 50)

# Eliminar ID y otras variables no relevantes
variables_eliminar = ['ID']
df_clean = df.drop(columns=[col for col in variables_eliminar if col in df.columns])

print(f"✓ Dataset final: {df_clean.shape[0]} filas, {df_clean.shape[1]} columnas")

# Variables para análisis
numeric_vars = ['Transportation_expense', 'Distance_Residence_Work', 'Service_time', 
                'Age', 'Work_load_Average_day', 'Hit_target', 'Disciplinary_failure', 
                'Son', 'Social_drinker', 'Social_smoker', 'Pet', 'Weight', 'Height', 
                'Body_mass_index', 'Education_numeric', 'Month_absence_order', 
                'Day_week_order', 'Seasons_order']

categorical_vars = ['Month_absence', 'Day_week', 'Seasons', 'Education']
target_var = 'Absenteeism_hours'

# Verificar que las variables existen en el dataset
numeric_vars = [var for var in numeric_vars if var in df_clean.columns]
categorical_vars = [var for var in categorical_vars if var in df_clean.columns]

print(f"✓ Variables numéricas para análisis: {len(numeric_vars)}")
print(f"✓ Variables categóricas para análisis: {len(categorical_vars)}")

# 4. ANÁLISIS DE NORMALIDAD
print("\n4. ANÁLISIS DE NORMALIDAD (Shapiro-Wilk)...")
print("-" * 50)

normality_results = []
for var in numeric_vars + [target_var]:
    if var in df_clean.columns:
        data = df_clean[var].dropna()
        if len(data) > 3:  # Mínimo requerido para Shapiro
            stat, p_value = shapiro(data)
            normality_results.append({
                'Variable': var,
                'Estadístico': round(stat, 6),
                'p_value': round(p_value, 6),
                'Normal': p_value > 0.05,
                'n': len(data)
            })

df_normality = pd.DataFrame(normality_results)
if not df_normality.empty:
    print(df_normality.to_string(index=False))
else:
    print("No se pudieron realizar tests de normalidad")

# 5. ANÁLISIS DE CORRELACIÓN (Spearman)
print("\n5. ANÁLISIS DE CORRELACIÓN (Spearman)...")
print("-" * 50)

correlation_results = []
for var in numeric_vars:
    if var in df_clean.columns:
        # Verificar que hay suficientes datos
        valid_data = df_clean[[var, target_var]].dropna()
        if len(valid_data) > 10:  # Mínimo para correlación
            corr, p_value = spearmanr(valid_data[var], valid_data[target_var])
            
            # Clasificar fuerza de correlación
            abs_corr = abs(corr)
            if abs_corr < 0.1:
                fuerza = "Muy débil"
            elif abs_corr < 0.3:
                fuerza = "Débil" 
            elif abs_corr < 0.5:
                fuerza = "Moderada"
            elif abs_corr < 0.7:
                fuerza = "Fuerte"
            else:
                fuerza = "Muy fuerte"
            
            direccion = "Positiva" if corr > 0 else "Negativa"
            
            correlation_results.append({
                'Variable': var,
                'Test': 'Spearman',
                'Correlación': round(corr, 6),
                'p_value': round(p_value, 6),
                'Significativa': p_value < 0.05,
                'Fuerza': fuerza,
                'Dirección': direccion,
                'n': len(valid_data)
            })

df_correlation_numeric = pd.DataFrame(correlation_results)
if not df_correlation_numeric.empty:
    print(df_correlation_numeric.to_string(index=False))
else:
    print("No se pudieron calcular correlaciones")

# 6. ANÁLISIS DE VARIABLES CATEGÓRICAS (Kruskal-Wallis)
print("\n6. ANÁLISIS DE VARIABLES CATEGÓRICAS (Kruskal-Wallis)...")
print("-" * 50)

categorical_results = []
for cat_var in categorical_vars:
    if cat_var in df_clean.columns:
        groups = [group[target_var].values for name, group in df_clean.groupby(cat_var) if len(group) > 5]
        
        if len(groups) > 1:
            try:
                stat, p_value = kruskal(*groups)
                categorical_results.append({
                    'Variable': cat_var,
                    'Test': 'Kruskal-Wallis H',
                    'Estadístico': round(stat, 6),
                    'p_value': round(p_value, 6),
                    'Significativa': p_value < 0.05,
                    'Número_Categorías': len(groups),
                    'n_total': sum(len(g) for g in groups)
                })
            except Exception as e:
                print(f"✗ Error en test para {cat_var}: {e}")

df_categorical = pd.DataFrame(categorical_results)
if not df_categorical.empty:
    print(df_categorical.to_string(index=False))
else:
    print("No se pudieron realizar tests para variables categóricas")

# 7. GRÁFICOS DE ANÁLISIS
print("\n7. GENERANDO GRÁFICOS DE ANÁLISIS...")
print("-" * 50)

try:
    # A. Heatmap de correlaciones
    plt.figure(figsize=(16, 14))
    corr_vars = [v for v in numeric_vars if v in df_clean.columns] + [target_var]
    if len(corr_vars) > 1:
        corr_matrix = df_clean[corr_vars].corr(method='spearman')
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

        sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0, 
                    square=True, fmt='.3f', cbar_kws={'shrink': 0.8})
        plt.title('MATRIZ DE CORRELACIÓN SPEARMAN\nVariables vs Ausentismo Laboral', 
                fontsize=16, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.savefig(f'{output_path}/correlation_heatmap.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("✓ Heatmap guardado: correlation_heatmap.png")
    
    # B. Gráfico de importancia de variables
    if not df_correlation_numeric.empty:
        plt.figure(figsize=(12, 10))
        corr_data = df_correlation_numeric.copy()
        corr_data['abs_corr'] = abs(corr_data['Correlación'])
        corr_data = corr_data.sort_values('abs_corr', ascending=True)

        colors = ['red' if sig else 'gray' for sig in corr_data['Significativa']]
        plt.barh(corr_data['Variable'], corr_data['abs_corr'], color=colors, alpha=0.7)
        plt.xlabel('Correlación Absoluta (Spearman)', fontsize=12)
        plt.title('IMPORTANCIA DE VARIABLES PARA PREDECIR AUSENTISMO\n(Rojo = Estadísticamente Significativo, p < 0.05)', 
                fontsize=14, fontweight='bold')
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{output_path}/variable_importance.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("✓ Gráfico de importancia guardado: variable_importance.png")
        
except Exception as e:
    print(f"✗ Error generando gráficos: {e}")

# 8. ESTADÍSTICAS DESCRIPTIVAS
print("\n8. ESTADÍSTICAS DESCRIPTIVAS...")
print("-" * 50)

try:
    analysis_vars = [v for v in numeric_vars + [target_var] if v in df_clean.columns]
    if analysis_vars:
        desc_stats = df_clean[analysis_vars].describe().T
        desc_stats['IQR'] = desc_stats['75%'] - desc_stats['25%']
        desc_stats['CV'] = (desc_stats['std'] / desc_stats['mean']).round(3)
        print(desc_stats.round(3))
    else:
        print("No hay variables numéricas para análisis")
except Exception as e:
    print(f"Error calculando estadísticas descriptivas: {e}")

# 9. EXPORTACIÓN DE RESULTADOS
print("\n9. EXPORTANDO RESULTADOS...")
print("-" * 50)

try:
    # Preparar datos para exportación
    resultados = {}
    
    if not df_normality.empty:
        resultados['Normalidad'] = df_normality
    
    if not df_correlation_numeric.empty:
        resultados['Correlacion_Numericas'] = df_correlation_numeric
    
    if not df_categorical.empty:
        resultados['Correlacion_Categoricas'] = df_categorical
    
    # Estadísticas descriptivas
    if not desc_stats.empty:
        resultados['Estadisticas_Descriptivas'] = desc_stats.reset_index().rename(columns={'index': 'Variable'})

    # Exportar en CSV y Parquet
    for nombre, dataframe in resultados.items():
        if not dataframe.empty:
            # CSV
            csv_path = f"{output_path}/{nombre}.csv"
            dataframe.to_csv(csv_path, index=False, encoding='utf-8-sig')
            print(f"✓ {nombre}.csv guardado")
            
            # Parquet
            parquet_path = f"{output_path}/{nombre}.parquet"
            dataframe.to_parquet(parquet_path, index=False)
            print(f"✓ {nombre}.parquet guardado")

    # Guardar dataset limpio
    df_clean.to_parquet(f"{output_path}/dataset_analisis_limpio.parquet", index=False)
    print("✓ Dataset limpio guardado: dataset_analisis_limpio.parquet")
    
except Exception as e:
    print(f"✗ Error exportando resultados: {e}")

# 10. RESUMEN EJECUTIVO DETALLADO (CORREGIDO)
print("\n" + "="*80)
print("RESUMEN EJECUTIVO DETALLADO")
print("="*80)

print(f"\n📊 DATASET ANALIZADO:")
print(f"   • Filas: {df_clean.shape[0]:,}")
print(f"   • Columnas: {df_clean.shape[1]}")
print(f"   • Variables numéricas analizadas: {len(numeric_vars)}")
print(f"   • Variables categóricas analizadas: {len(categorical_vars)}")

# Normalidad
if not df_normality.empty:
    normales = df_normality['Normal'].sum()
    no_normales = len(df_normality) - normales
    print(f"\n📈 NORMALIDAD DE VARIABLES:")
    print(f"   • Variables normales: {normales} ({normales/len(df_normality)*100:.1f}%)")
    print(f"   • Variables no normales: {no_normales} ({no_normales/len(df_normality)*100:.1f}%)")

# Correlaciones
if not df_correlation_numeric.empty:
    sig_numeric = df_correlation_numeric['Significativa'].sum()
    print(f"\n🔗 CORRELACIONES SIGNIFICATIVAS (p < 0.05):")
    print(f"   • Variables numéricas significativas: {sig_numeric}/{len(df_correlation_numeric)}")

if not df_categorical.empty:
    sig_categorical = df_categorical['Significativa'].sum()
    print(f"   • Variables categóricas significativas: {sig_categorical}/{len(df_categorical)}")

# Variables importantes (CORRECCIÓN DEL ERROR)
if not df_correlation_numeric.empty:
    print(f"\n🎯 VARIABLES MÁS IMPORTANTES:")
    
    # Crear columna temporal para el valor absoluto
    df_temp = df_correlation_numeric.copy()
    df_temp['abs_correlacion'] = df_temp['Correlación'].abs()
    
    # Obtener top 5 por correlación absoluta
    top_vars = df_temp.nlargest(5, 'abs_correlacion')[['Variable', 'Correlación', 'Significativa']]
    
    for _, row in top_vars.iterrows():
        sig_symbol = "✓" if row['Significativa'] else "✗"
        print(f"   • {row['Variable']}: {row['Correlación']:.3f} [{sig_symbol}]")

# Variables con correlación moderada/fuerte
if not df_correlation_numeric.empty:
    strong_vars = df_correlation_numeric[
        df_correlation_numeric['Fuerza'].isin(['Moderada', 'Fuerte', 'Muy fuerte']) & 
        df_correlation_numeric['Significativa']
    ]
    
    if not strong_vars.empty:
        print(f"\n💪 VARIABLES CON CORRELACIÓN MODERADA/FUERTE:")
        for _, row in strong_vars.iterrows():
            print(f"   • {row['Variable']}: ρ = {row['Correlación']:.3f} ({row['Fuerza']}, {row['Dirección']})")

print(f"\n💾 RESULTADOS EXPORTADOS:")
print(f"   • Ubicación: {output_path}")
print(f"   • Formatos: CSV y Parquet")
print(f"   • Archivos generados: Normalidad, Correlaciones, Estadísticas Descriptivas")

print(f"\n🎯 HALLAZGOS PRINCIPALES:")
if not df_correlation_numeric.empty:
    # Variable con mayor correlación positiva
    max_pos = df_correlation_numeric[df_correlation_numeric['Correlación'] > 0]
    if not max_pos.empty:
        top_pos = max_pos.nlargest(1, 'Correlación').iloc[0]
        print(f"   • Mayor correlación positiva: {top_pos['Variable']} (ρ = {top_pos['Correlación']:.3f})")
    
    # Variable con mayor correlación negativa
    max_neg = df_correlation_numeric[df_correlation_numeric['Correlación'] < 0]
    if not max_neg.empty:
        top_neg = max_neg.nsmallest(1, 'Correlación').iloc[0]
        print(f"   • Mayor correlación negativa: {top_neg['Variable']} (ρ = {top_neg['Correlación']:.3f})")

print(f"\n📋 RECOMENDACIONES:")
print(f"   • Enfocar intervenciones en variables con correlación significativa")
print(f"   • Considerar el contexto temporal (mes, día, estación) en análisis futuros")
print(f"   • Validar hallazgos con modelos predictivos adicionales")

print("\n" + "="*80)
print("ANÁLISIS COMPLETADO EXITOSAMENTE")
print("="*80)

ANÁLISIS ESTADÍSTICO DE AUSENTISMO LABORAL

1. CARGANDO Y VERIFICANDO DATOS...
--------------------------------------------------
✓ Dataset cargado correctamente: 706 filas, 25 columnas

2. VERIFICACIÓN DE TIPOS DE DATOS...
--------------------------------------------------
Tipos de datos actuales:
  ID: int64
  Reason_absence: string
  Month_absence: string
  Day_week: string
  Seasons: string
  Transportation_expense: float64
  Distance_Residence_Work: float64
  Service_time: float64
  Age: float64
  Work_load_Average_day: float64
  Hit_target: float64
  Disciplinary_failure: int64
  Education: string
  Son: int64
  Social_drinker: int64
  Social_smoker: int64
  Pet: int64
  Weight: float64
  Height: float64
  Body_mass_index: float64
  Absenteeism_hours: float64
  Education_numeric: int64
  Month_absence_order: int64
  Day_week_order: int64
  Seasons_order: int64
✓ Todos los tipos de datos son correctos

3. PREPARACIÓN DE VARIABLES...
------------------------------------------------